In [1]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

In [3]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print(x_train.shape)
print(x_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1898s 11us/step
(50000, 32, 32, 3)
(10000, 32, 32, 3)


In [4]:
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

In [5]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))

train_ds = train_ds.shuffle(10000)

train_ds = train_ds.map(
    lambda x, y: (
        tf.image.resize(tf.cast(x, tf.float32), (IMG_SIZE, IMG_SIZE)),
        y
    ),
    num_parallel_calls=tf.data.AUTOTUNE
)

test_ds = test_ds.map(
    lambda x, y: (
        tf.image.resize(tf.cast(x, tf.float32), (IMG_SIZE, IMG_SIZE)),
        y
    ),
    num_parallel_calls=tf.data.AUTOTUNE
)

train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [6]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model

vgg_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

vgg_base.trainable = False

inputs = Input(shape=(224, 224, 3))

x = vgg_base(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

outputs = Dense(10, activation='softmax')(x)

vgg_model = Model(inputs, outputs)

vgg_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [8]:
history_vgg = vgg_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=3
)

Epoch 1/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 331s 212ms/step - accuracy: 0.7672 - loss: 0.6772 - val_accuracy: 0.7979 - val_loss: 0.5870
Epoch 2/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 330s 211ms/step - accuracy: 0.7783 - loss: 0.6400 - val_accuracy: 0.8054 - val_loss: 0.5646
Epoch 3/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 330s 211ms/step - accuracy: 0.7871 - loss: 0.6205 - val_accuracy: 0.8103 - val_loss: 0.5528


In [9]:
from tensorflow.keras.applications import EfficientNetB0

efficient_base = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

efficient_base.trainable = False

inputs = Input(shape=(224, 224, 3))

x = efficient_base(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

outputs = Dense(10, activation='softmax')(x)

efficient_model = Model(inputs, outputs)

efficient_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [10]:
history_efficient = efficient_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=3
)

Epoch 1/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 121s 59ms/step - accuracy: 0.8460 - loss: 0.4597 - val_accuracy: 0.8957 - val_loss: 0.3102
Epoch 2/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 65s 41ms/step - accuracy: 0.8791 - loss: 0.3514 - val_accuracy: 0.9047 - val_loss: 0.2832
Epoch 3/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 64s 41ms/step - accuracy: 0.8899 - loss: 0.3217 - val_accuracy: 0.9051 - val_loss: 0.2813


In [11]:
vgg_loss, vgg_accuracy = vgg_model.evaluate(test_ds)

efficient_loss, efficient_accuracy = efficient_model.evaluate(test_ds)

print("VGG16 Accuracy:", vgg_accuracy * 100)
print("EfficientNetB0 Accuracy:", efficient_accuracy * 100)

313/313 ━━━━━━━━━━━━━━━━━━━━ 56s 180ms/step - accuracy: 0.8103 - loss: 0.5528
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.9051 - loss: 0.2813
VGG16 Accuracy: 81.02999925613403
EfficientNetB0 Accuracy: 90.50999879837036


In [12]:
results = pd.DataFrame({
    'Model': ['VGG16', 'EfficientNetB0'],
    'Accuracy (%)': [
        vgg_accuracy * 100,
        efficient_accuracy * 100
    ]
})

results

,Model,Accuracy (%)
0,VGG16,81.029999
1,EfficientNetB0,90.509999
